# Module 1 · Lesson 04: Compare Models Side by Side

A key skill for any AI engineer is **choosing the right model** for the job.
In this notebook we send the *same* prompt to multiple models and compare results.

## What you will learn
1. How to call different providers with a **unified interface**
2. Compare **quality, speed, and cost** across models
3. Build a reusable comparison framework
4. Understand the **speed vs quality vs cost** trade-off

In [ ]:
import os

from pathlib import Path
from dotenv import  load_dotenv
from IPython.display import display, Markdown, clear_output

load_dotenv(Path.cwd().parent / "API_Verifications/.env")
    

from openai import OpenAI

openai_client = OpenAI()
OPENAI_MODEL = "gpt-4o-mini"
if openai_client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")

try:
    import anthropic
    if os.getenv("ANTHROPIC_API_KEY"):
        claude_client = anthropic.Anthropic()
        ANTH_MODEL = "claude-sonnet-4-6"
        print(f"Antropic client ready - using model {ANTH_MODEL}")
    else:
        print("Anthropic AI isn't ready")    
except ImportError:
    print("Anthopic not installed") 


Antropic client ready - using model claude-sonnet-4-6
OpenAI client ready - using model {OPENAI_MODEL}


---
## 1. Model Registry & Pricing

First, let's define the models and their costs (per 1M tokens, early 2025):

LiteLLM provides a **unified interface** to call any LLM provider with the same code.
Instead of learning separate SDKs, you write one `completion()` call and LiteLLM handles the rest.

```bash
pip install litellm openai anthropic
```

In [2]:
import litellm
from litellm import completion, completion_cost, model_cost

MODELS = [
    "gpt-4o-mini",
    "gpt-4o",
    "gpt-5.2",
    "claude-haiku-4-5",
    "claude-opus-4-6"
]

pricing_md = []

for model in MODELS:
    try:
        info = model_cost[model]
        input_price = info["input_cost_per_token"] * 1_000_000
        ouput_price = info["output_cost_per_token"] * 1_000_000
        pricing_md.append(
            f"- **{model}** -- Input: `${input_price:.2f}` | Output: `${ouput_price:.2f}`"
        )
    except KeyError:
        pricing_md.append(f"- **{model}** -- Pricing not found")

display(Markdown("### Model Pricing (per 1M tokens)\n" + "\n".join(pricing_md) ))

### Model Pricing (per 1M tokens)
- **gpt-4o-mini** -- Input: `$0.15` | Output: `$0.60`
- **gpt-4o** -- Input: `$2.50` | Output: `$10.00`
- **gpt-5.2** -- Input: `$1.75` | Output: `$14.00`
- **claude-haiku-4-5** -- Input: `$1.00` | Output: `$5.00`
- **claude-opus-4-6** -- Input: `$5.00` | Output: `$25.00`

---
### Sending the Same Prompt to Every Model

With `litellm.completion()` we can send the exact same prompt to OpenAI and Anthropic models
using **identical code**. LiteLLM translates the call to each provider's native API behind the scenes.

In [3]:
prompt = "Explain what an API is in 2-3 sentences"

for model in MODELS:
    try:
        response = completion(
            model= model,
            messages= [{"role" : "user", "content": prompt}],
            max_tokens= 200
        )
        reply = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        cost = completion_cost(completion_response= response)

        md = f"""### {model}
        **{reply}**
         - Input tokens: `{input_tokens}` | Output tokens: `{output_tokens}` | Cost: `${cost:.6f}` """
        
        display(Markdown(md))
    except Exception as e:
        print(f"Error: {e}")

### gpt-4o-mini
        **An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods and data formats that applications can use to request and exchange information, enabling developers to integrate functionalities and access external services seamlessly. By using APIs, developers can build applications that leverage the capabilities of other software, platforms, or services without needing to understand their internal workings.**
         - Input tokens: `18` | Output tokens: `82` | Cost: `$0.000052` 

### gpt-4o
        **An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods and data formats that developers can use to interact with an application, service, or system. By providing a structured way for programs to interact, APIs enable the integration and functionality enhancement of applications without the need for developers to understand the underlying codebase.**
         - Input tokens: `18` | Output tokens: `79` | Cost: `$0.000835` 

### gpt-5.2
        **An API (Application Programming Interface) is a set of rules and endpoints that lets one software system communicate with another. It defines what requests you can make, what data you must send, and what responses you’ll receive, so developers can use a service without needing to know how it’s built internally.**
         - Input tokens: `17` | Output tokens: `63` | Cost: `$0.000912` 

### claude-haiku-4-5
        **# What is an API?

An API (Application Programming Interface) is a set of rules and tools that allows different software applications to communicate and share data with each other. It acts like a bridge between programs, letting one application request information or services from another without needing to know how the other application works internally. For example, when you check the weather in your phone's calendar app, it's using a weather API to fetch that data from a weather service.**
         - Input tokens: `20` | Output tokens: `97` | Cost: `$0.000505` 

### claude-opus-4-6
        **An **API (Application Programming Interface)** is a set of rules and protocols that allows different software applications to communicate with each other. It defines the methods and data formats that programs can use to request and exchange information, acting as an intermediary between systems. For example, when a weather app on your phone displays current conditions, it's likely using an API to fetch that data from a remote weather service.**
         - Input tokens: `20` | Output tokens: `85` | Cost: `$0.002225` 

---
### Multi-Turn Conversation with Cost Tracking

LiteLLM also supports multi-turn conversations. We can track the cumulative cost across turns.

In [ ]:
model = OPENAI_MODEL
total_cost = 0.0

conversation = [
    {"role": "user", "content": "Hello. My name is Panos and i am software engineer."},
]

# Turn 1
response_turn1 = completion(model= model, messages= conversation, max_tokens= 150)
assistant_reply1 = response_turn1.choices[0].message.content
input_tokens = response_turn1.usage.prompt_tokens
output_tokens = response_turn1.usage.completion_tokens
total_cost += completion_cost(completion_response= response_turn1)

display(Markdown(f"""
**User:** Hello. My name is Panos and i am software engineer.
                 
**Assistant:** {assistant_reply1}      

-- Input tokens: {input_tokens}
    -- Output tokens: {output_tokens}
    **Cost:** `${total_cost:.6f}`           
"""))




**User:** Hello. My name is Panos and i am software engineer.

**Assistant:** Hello, Panos! It's great to meet you. As a software engineer, what kind of projects are you working on, or what technologies are you interested in?      

-- Input tokens: 20
    -- Output tokens: 85
    **Cost:** `$0.000023`           


In [ ]:
# Turn 2
conversation.append({"role": "assistant", "content": assistant_reply1})
conversation.append({"role": "user", "content": "What is my name and what do I do?"})
response_turn2 = completion(model= model, messages= conversation, max_tokens= 150)
assistant_reply2 = response_turn2.choices[0].message.content
input_tokens = response_turn2.usage.prompt_tokens
output_tokens = response_turn2.usage.completion_tokens
total_cost += completion_cost(completion_response= response_turn2)

display(Markdown(f"""
**User:** What is my name and what do I do?
                 
**Assistant:** {assistant_reply1}      

-- Input tokens: {input_tokens}\t
-- Output tokens: {output_tokens}\t
**Cost:** `${total_cost:.6f}`           
"""))


**User:** What is my name and what do it do?

**Assistant:** Hello, Panos! It's great to meet you. As a software engineer, what kind of projects are you working on, or what technologies are you interested in?      

-- Input tokens: 20	
-- Output tokens: 85	
**Cost:** `$0.000045`           


---
### Cost Comparison for the Same Prompt

Let's send the same creative prompt to every model and compare costs side by side, sorted cheapest first.

In [7]:
# A haiku has 3 lines with a specific syllable pattern
# Line 1 → 5 syllables
# Line 2 → 7 syllables
# Line 3 → 5 syllables

prompt = "Write a haiku about programming."
results = []

# Run prompt across models
for model in MODELS:
    try:
        response = completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50,
        )
        cost = completion_cost(completion_response=response)
        reply = response.choices[0].message.content
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens
        results.append({"model": model, "cost": cost, "reply": reply, "input_tokens": input_tokens, 
                        "output_tokens": output_tokens})
    except Exception as e:
        results.append({"model": model, "cost": None, "reply": f"Error: {e}", "input_tokens" : None,
        "output_tokens": None})

# Sort by cost (cheapest first)
results.sort(key=lambda x: x["cost"] if x["cost"] is not None else float("inf"))

# Display results using Markdown
display(Markdown("## Cost comparison for the same prompt"))

for r in results:
    cost_str = f"${r['cost']:.6f}" if r["cost"] is not None else "N/A"

    display(Markdown(f"""
### {r['model']}
- Cost: **{cost_str}**
- Input tokens: {r['input_tokens']}
- Output tokens: {r['output_tokens']}

**Response**
```text
{r['reply']}
```
"""))

## Cost comparison for the same prompt


### gpt-4o-mini
- Cost: **$0.000015**
- Input tokens: 14
- Output tokens: 21

**Response**
```text
Code flows like a stream,  
Logic dances on the screen,  
Dreams in lines of text.
```



### claude-haiku-4-5
- Cost: **$0.000154**
- Input tokens: 14
- Output tokens: 28

**Response**
```text
# Code in the Dark

Loops iterate on,
Bugs hide in nested brackets—
Coffee grows me wings.
```



### gpt-4o
- Cost: **$0.000205**
- Input tokens: 14
- Output tokens: 17

**Response**
```text
Lines of code connect,  
Logic dances through the night,  
Ideas take flight.
```



### gpt-5.2
- Cost: **$0.000331**
- Input tokens: 13
- Output tokens: 22

**Response**
```text
Silent keys echo,  
Logic blooms in lines of code—  
Bugs fade with sunrise.
```



### claude-opus-4-6
- Cost: **$0.000970**
- Input tokens: 14
- Output tokens: 36

**Response**
```text
# A Programming Haiku

*Code flows line by line*
*Bugs hide in the logic's maze*
*One fix breaks the rest*
```


---
## 2. Unified Call Function

To compare models fairly, we need a function that calls *any* provider and returns standardized results:

In [11]:
import time

# Model registry for direct SDK calls (Sections 2-5)
MODEL_REGISTRY = {
    "gpt-4o-mini":       {"provider": "openai",    "input": 0.15,  "output": 0.60},
    "gpt-4o":            {"provider": "openai",    "input": 2.50,  "output": 10.00},
    "claude-sonnet-4-6": {"provider": "anthropic", "input": 3.00,  "output": 15.00},
    "claude-haiku-4-5":  {"provider": "anthropic", "input": 1.00,  "output": 5.00},
    "claude-opus-4-6":   {"provider": "anthropic", "input": 5.00,  "output": 25.00},
}

def estimate_cost(model, input_tokens, output_tokens):
    info = MODEL_REGISTRY[model]
    return (input_tokens / 1_000_000) * info["input"] + (output_tokens / 1_000_000) * info["output"]

def call_model(prompt: str, model: str, temperature: float = 0.7) -> dict:
    """Call any model and return standardized result dict."""
    info = MODEL_REGISTRY[model]
    start = time.perf_counter()

    if info["provider"] == "openai":
        r = openai_client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=temperature
        )
        text = r.choices[0].message.content
        in_tok, out_tok = r.usage.prompt_tokens, r.usage.completion_tokens

    elif info["provider"] == "anthropic":
        r = claude_client.messages.create(
            model=model, max_tokens=200, temperature=temperature,
            messages=[{"role": "user", "content": prompt}]
        )
        text = r.content[0].text
        in_tok, out_tok = r.usage.input_tokens, r.usage.output_tokens

    elapsed = time.perf_counter() - start
    cost = estimate_cost(model, in_tok, out_tok)

    return {
        "model": model, "provider": info["provider"],
        "text": text, "input_tokens": in_tok, "output_tokens": out_tok,
        "latency_ms": round(elapsed * 1000), "cost_usd": cost
    }

---
## 3. Head-to-Head Comparison

Let's compare all available models on the **same prompt**:

In [12]:
from IPython.display import display, Markdown

# Run comparison
test_prompt = "Explain what a REST API is to a junior developer. Be concise."

display(Markdown(f"##  Model Comparison\n**Prompt:** \"{test_prompt}\"\n---"))

results = []

for model_name in MODEL_REGISTRY:
    try:
        result = call_model(test_prompt, model_name)
        results.append(result)

        # IMPORTANT: do NOT wrap result['text'] in ``` or <pre>
        md = f"""
### 🤖 {model_name} ({result['provider']})

- **Latency:** `{result['latency_ms']} ms`
- **Tokens:** `{result['input_tokens']} in / {result['output_tokens']} out`
- **Cost:** `${result['cost_usd']:.6f}`

---

{result['text']}

---
"""
        display(Markdown(md))

    except Exception as e:
        display(Markdown(f"###  {model_name}\n`Error: {e}`"))

##  Model Comparison
**Prompt:** "Explain what a REST API is to a junior developer. Be concise."
---


### 🤖 gpt-4o-mini (openai)

- **Latency:** `3455 ms`
- **Tokens:** `21 in / 196 out`
- **Cost:** `$0.000121`

---

A REST API (Representational State Transfer Application Programming Interface) is a set of rules that allows different software systems to communicate over the internet using standard HTTP methods like GET, POST, PUT, and DELETE. 

In a REST API:

- **Resources**: Everything is treated as a resource, which can be identified by a URL (e.g., `/users`, `/products`).
- **Stateless**: Each request from a client to the server must contain all the information needed to understand and process it, meaning the server doesn’t store any session information.
- **Use of Standard Methods**: 
  - **GET**: Retrieve data.
  - **POST**: Create new data.
  - **PUT**: Update existing data.
  - **DELETE**: Remove data.

REST APIs typically return data in JSON or XML format, making it easy to work with in various programming languages. They are widely used for web services due to their simplicity and scalability.

---



### 🤖 gpt-4o (openai)

- **Latency:** `1706 ms`
- **Tokens:** `21 in / 108 out`
- **Cost:** `$0.001133`

---

A REST API (Representational State Transfer Application Programming Interface) is a way for different software applications to communicate over the internet using HTTP requests. It allows you to perform operations like retrieving, adding, updating, or deleting data on a server. REST APIs use standard HTTP methods like GET, POST, PUT, and DELETE, and they often work with JSON or XML data formats. They're designed to be stateless, meaning each request from a client contains all the information needed to understand and process it, which makes them scalable and easy to use.

---



### 🤖 claude-sonnet-4-6 (anthropic)

- **Latency:** `4999 ms`
- **Tokens:** `23 in / 200 out`
- **Cost:** `$0.003069`

---

# What is a REST API?

A **REST API** is a way for two applications to talk to each other over the internet using standard HTTP requests.

## The Key Idea
Think of it like a **waiter in a restaurant**:
- You (the client) make a **request**
- The waiter (the API) carries it to the kitchen (the server)
- The kitchen sends back a **response**

---

## How it Works

You communicate using **HTTP methods** that map to actions:

| Method | Action | Example |
|--------|--------|---------|
| `GET` | Read data | Fetch a user profile |
| `POST` | Create data | Register a new user |
| `PUT` | Update data | Edit a profile |
| `DELETE` | Delete data | Remove a user |

---

## A Simple Example

To get a user's data, you might

---



### 🤖 claude-haiku-4-5 (anthropic)

- **Latency:** `2069 ms`
- **Tokens:** `23 in / 200 out`
- **Cost:** `$0.001023`

---

# REST API Explained

A **REST API** is a set of rules for building web services that let different applications communicate and share data over the internet.

## Key Points

**REST** = Representational State Transfer

**Core Idea:** Use standard HTTP methods to perform actions on resources (data):
- **GET** - retrieve data
- **POST** - create data
- **PUT** - update data
- **DELETE** - remove data

## Simple Example

Instead of custom commands, you use URLs:
```
GET /api/users/5          → Get user with ID 5
POST /api/users           → Create a new user
PUT /api/users/5          → Update user 5
DELETE /api/users/5       → Delete user 5
```

## Why It Matters

- **Standard** - everyone follows the same conventions
- **Simple** - uses HTTP, which browsers already understand

---



### 🤖 claude-opus-4-6 (anthropic)

- **Latency:** `5959 ms`
- **Tokens:** `23 in / 200 out`
- **Cost:** `$0.005115`

---

# REST API — A Simple Explanation

A **REST API** is a way for two applications to communicate over the internet using standard **HTTP** methods — the same protocol your browser uses.

## Core Idea
Your app sends a **request** to a URL (called an **endpoint**), and the server sends back a **response** (usually JSON).

## The Main HTTP Methods

| Method   | Action   | Example                  |
|----------|----------|--------------------------|
| `GET`    | Read     | Fetch a list of users    |
| `POST`   | Create   | Add a new user           |
| `PUT`    | Update   | Edit an existing user    |
| `DELETE` | Remove   | Delete a user            |

## Quick Example

```
GET https://api.example.com/users/42
```
→ Returns the user with ID 42 as JSON:

---


---
## 4. Summary Table

In [13]:
# ── Summary table ─────────────────────────────────────
if results:
    header =  "| Model | Provider | Latency | Tokens | Cost |\n"
    header += "|-------|----------|---------|--------|------|\n"
    rows = ""
    for r in results:
        rows += (f"| {r['model']} | {r['provider']} | {r['latency_ms']} ms | "
                 f"{r['input_tokens']}+{r['output_tokens']} | ${r['cost_usd']:.6f} |\n")
    display(Markdown(header + rows))

    # Cheapest and fastest
    cheapest = min(results, key=lambda x: x['cost_usd'])
    fastest  = min(results, key=lambda x: x['latency_ms'])
    print(f"\n Cheapest: {cheapest['model']} (${cheapest['cost_usd']:.6f})")
    print(f" Fastest:  {fastest['model']} ({fastest['latency_ms']} ms)")

| Model | Provider | Latency | Tokens | Cost |
|-------|----------|---------|--------|------|
| gpt-4o-mini | openai | 3455 ms | 21+196 | $0.000121 |
| gpt-4o | openai | 1706 ms | 21+108 | $0.001133 |
| claude-sonnet-4-6 | anthropic | 4999 ms | 23+200 | $0.003069 |
| claude-haiku-4-5 | anthropic | 2069 ms | 23+200 | $0.001023 |
| claude-opus-4-6 | anthropic | 5959 ms | 23+200 | $0.005115 |



 Cheapest: gpt-4o-mini ($0.000121)
 Fastest:  gpt-4o (1706 ms)


---
## 5. Multiple Test Prompts

A single prompt isn't enough — let's test across different task types:

In [ ]:
from IPython.display import display, Markdown

# ── Multi-test comparison ─────────────────────────────
test_suite = [
    ("Factual",    "What is the speed of light in km/s?"),
    ("Creative",   "Write a haiku about programming."),
    ("Analytical", "What are 3 pros and 3 cons of microservices?"),
    ("Reasoning",  "If all roses are flowers and some flowers fade quickly, can we conclude that some roses fade quickly?"),
]

for test_name, prompt in test_suite:
    display(Markdown(f"""
##  Test: {test_name}
**Prompt:** {prompt}
---
"""))

    for model_name in MODEL_REGISTRY:
        try:
            r = call_model(prompt, model_name, temperature=0.3)

            # Preview (first 200 chars) but keep it markdown-safe and readable
            preview = r["text"][:200] + ("…" if len(r["text"]) > 200 else "")
            preview = preview.replace("\n", " ")  # keep preview on one line

            display(Markdown(f"""
###  {r['model']}
- **Latency:** `{r['latency_ms']} ms`
- **Cost:** `${r['cost_usd']:.6f}`

**Preview:** {preview}

<details>
<summary>Show full response</summary>

{r['text']}

</details>

---
"""))
        except Exception as e:
            display(Markdown(f"###  {model_name}\n`Error: {e}`\n---"))

---
## Key Takeaways 📝

| Insight | Detail |
|---------|--------|
| **Speed ≠ Quality** | Faster models may give shorter, simpler answers |
| **Cost varies 10–100×** | gpt-4o-mini costs ~16× less than gpt-4o |
| **Use the cheapest model that works** | Start with mini/haiku, upgrade if quality is insufficient |
| **Multi-provider = resilience** | If one API is down, switch to another |
| **Always benchmark** | Don't assume — measure quality on *your* specific tasks |
| **Alternative providers** | Same SDK works with Groq, Ollama, and others |

---
**Next:** `05_token_explorer.ipynb` — Deep dive into tokenization and cost calculation